# OmniQuant-Apex: Fine-tuning on Google Colab

Fine-tunes pre-trained ULEP + MR-GWD models on your video dataset.
Uses free T4 GPU on Colab.

## Steps:
1. Upload your video files to a folder
2. Run all cells
3. Download the trained ONNX models
4. Load them in the Rust engine

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install torch torchvision transformers diffusers accelerate lpips opencv-python imageio

In [ ]:
# Clone the OmniQuant-Apex repo (or mount your Google Drive with the code)
# Option A: Clone from repo
# !git clone https://github.com/your-org/omniaquant-apex.git
# %cd omniaquant-apex

# Option B: Mount Google Drive (if you uploaded the code there)
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/omniaquant-apex

In [ ]:
# Download pre-trained weights
!python train/download_pretrained.py --output onnx_models

In [ ]:
# Upload your video dataset
# Option A: Upload via Colab file picker
from google.colab import files
import os
import zipfile

os.makedirs('data/videos', exist_ok=True)
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('data/videos')
    else:
        os.rename(filename, f'data/videos/{filename}')
print(f"Uploaded {len(uploaded)} files")

In [ ]:
# Fine-tune the models
# This uses the pre-trained weights as initialization and fine-tunes on your data
!python train/train_pipeline.py \
  --data data/videos \
  --epochs 50 \
  --batch-size 4 \
  --lr 1e-5 \
  --latent-dim 512 \
  --gradient-accumulation 4 \
  --warmup-epochs 3 \
  --output-dir checkpoints \
  --onnx-dir onnx_models_finetuned

In [ ]:
# Evaluate on validation set
import torch
import sys
sys.path.insert(0, '.')

from ulep.model import ULEP
from mrgwd.model import MRGWD
from train.train_pipeline import VideoFrameDataset, OmniQuantLoss
from torch.utils.data import DataLoader

# Load best checkpoint
ckpt = torch.load('checkpoints/best.pt', map_location='cpu')

# Build models
ulep = ULEP(latent_dim=512, use_pretrained=False).cuda()
mrgwd = MRGWD(latent_dim=512, output_size=(256, 256), use_vae=False).cuda()

# Load weights
ulep.encode_head.load_state_dict(ckpt['ulep_encode_head'])
ulep.predictor_head.load_state_dict(ckpt['ulep_predictor_head'])
mrgwd.latent_synth.projector.load_state_dict(ckpt['latent_synth_projector'])
mrgwd.upsample_net.load_state_dict(ckpt['upsample_net'])

ulep.eval()
mrgwd.eval()

print(f"Best checkpoint loaded (epoch {ckpt['epoch']}, step {ckpt['global_step']})")
print(f"Validation loss: {ckpt['metrics']}")

In [ ]:
# Run inference demo to verify quality
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Load a test frame
dataset = VideoFrameDataset('data/videos', seq_len=4, max_videos=1)
batch = dataset[0].unsqueeze(0).cuda()  # (1, 4, 3, H, W)

# Encode + decode
with torch.no_grad():
    z_t = ulep.encode_trainable(batch[0, 0])
    frame_hat = mrgwd.synthesize(z_t)

# Convert to images
def tensor_to_img(t):
    t = (t.clamp(-1, 1) + 1) / 2
    return t.permute(1, 2, 0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(tensor_to_img(batch[0, 0]))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(tensor_to_img(frame_hat))
axes[1].set_title('Reconstructed')
axes[1].axis('off')
plt.tight_layout()
plt.savefig('comparison.png', dpi=150)
plt.show()

In [ ]:
# Download the fine-tuned ONNX models
import zipfile
import os

# Zip the models for download
with zipfile.ZipFile('onnx_models_finetuned.zip', 'w') as z:
    for root, dirs, files in os.walk('onnx_models_finetuned'):
        for f in files:
            z.write(os.path.join(root, f))

from google.colab import files
files.download('onnx_models_finetuned.zip')
print("Download started! Extract and place in your Rust project's onnx_models/ directory.")